In [ ]:
# Last Update: 7/4/2026 ; 3:18pm

In [3]:
# This is comprehensive TSFNet code, please set the paths basede on your system
import os
import random
import gc
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
import warnings
warnings.filterwarnings('ignore')

# CONFIGURATION
#PROCESSED_DATA_PATH = r"E:\kntu\Thesis\codes\Results\MULTIMODAL_PREPROCESSED"
PROCESSED_DATA_PATH = r"F:\preprocessing-history-thesis\7-3-2026 11=16 pm"

ALL_SUBJECTS = list(range(1, 27))
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

shuffled_subjects = ALL_SUBJECTS.copy()
random.shuffle(shuffled_subjects)

n_total = len(shuffled_subjects)
n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VAL_RATIO)
n_test = n_total - n_train - n_val

TRAIN_SUBS = shuffled_subjects[:n_train]
VAL_SUBS = shuffled_subjects[n_train:n_train + n_val]
TEST_SUBS = shuffled_subjects[n_train + n_val:]

class MMapHybridDataset(Dataset): 

    def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=2000):
        super().__init__()
        self.dataset_name = dataset_name
        self.valid_files = []
        self.total_windows = 0
        
        print(f"Scanning {dataset_name} - {len(subs_list)} subjects on Drive D...")
        
        # checnig subjects' quality
        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
            
            if not os.path.exists(sub_folder):
                continue
                
            eeg_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_eeg_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_eeg.npy")
            ] if os.path.exists(f)), None)
            
            nirs_file = next((f for f in [
                os.path.join(sub_folder, f"{subject}_fnirs_windows.npy"), 
                os.path.join(sub_folder, f"{subject}_fnirs.npy")
            ] if os.path.exists(f)), None)
            
            labels_file = os.path.join(sub_folder, f"{subject}_labels.npy")
            
            if eeg_file and nirs_file and os.path.exists(labels_file):
                try:
                    lbls = np.load(labels_file).astype(np.int64)
                    n_windows = lbls.shape[0]
                    
                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - Insufficient windows ({n_windows} < {min_windows_threshold}). Check task markers!")
                        continue
                    
                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject} verified: {n_windows} windows")
                    
                except Exception as e:
                    print(f"Error scanning {subject}: {str(e)}")
                    continue
                    
        if self.total_windows == 0:
            raise ValueError(f"No valid preprocessed data found for {dataset_name} after filtering!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)
        
        self.mmap_eeg_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_e.npy")
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_n.npy")
        
        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                try: os.remove(f_path)
                except: pass
            with open(f_path, 'wb') as f:
                pass 
        
        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', shape=(self.total_windows, 28, 600))
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', shape=(self.total_windows, 2, 100, 16, 16))
        self.y = np.zeros(self.total_windows, dtype='int64')
        
        print(f"Streaming clean data to disk memory-map ({dataset_name})...")
        for f_info in self.valid_files:
            s, e = f_info['start_idx'], f_info['end_idx']
            self.X_eeg[s:e] = np.load(f_info['eeg_path'])
            self.X_nirs[s:e] = np.load(f_info['nirs_path'])
            self.y[s:e] = f_info['labels']
            gc.collect()
            
        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"--> {dataset_name} Ready on Disk (CLEAN)! Total Windows: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor
        
    def cleanup(self):
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try: os.remove(f)
                except: pass

# MODEL COMPONENTS (TSFNet)
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): 
        return x + self.pe[:, :x.size(1), :]

class CrossAttention(nn.Module):
    def __init__(self, d_model, nhead=4, dropout=0.5):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, query, key, value):
        out, _ = self.attn(query, key, value)
        return self.norm(query + self.dropout(out))

class TSFNet_With_fNIRS_2D(nn.Module):
    def __init__(self, eeg_ch=28, num_classes=3, dropout=0.5):
        super(TSFNet_With_fNIRS_2D, self).__init__()
        
        # EEG branch
        self.eeg_conv1 = nn.Conv1d(eeg_ch, 32, kernel_size=12, stride=6)
        self.eeg_bn1 = nn.BatchNorm1d(32)
        self.eeg_drop1 = nn.Dropout(dropout)
        self.eeg_conv2 = nn.Conv1d(32, 64, kernel_size=6, stride=2)
        self.eeg_bn2 = nn.BatchNorm1d(64)
        self.eeg_drop2 = nn.Dropout(dropout)
        self.eeg_pool = nn.AdaptiveAvgPool1d(1)
        self.eeg_fc = nn.Linear(64, 64)
        
        # fNIRS branch
        self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        self.nirs_bn1 = nn.BatchNorm3d(16)
        self.nirs_drop1 = nn.Dropout(dropout)
        self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        self.nirs_bn2 = nn.BatchNorm3d(32)
        self.nirs_drop2 = nn.Dropout(dropout)
        self.nirs_spatial_pool = nn.AdaptiveAvgPool3d((None, 1, 1)) 
        self.nirs_time_pool = nn.AdaptiveAvgPool1d(1)               
        self.nirs_fc = nn.Linear(32, 64)
        
        # EFGF
        self.efgf_eeg = nn.Linear(64, 64)
        self.efgf_nirs = nn.Linear(64, 64)
        self.efgf_gate = nn.Linear(64, 1)
        self.alpha = nn.Parameter(torch.tensor(0.5))
        
        # CAFE
        self.pos_encoder = PositionalEncoding(64, max_len=100)
        self.cafe_attn = CrossAttention(64, nhead=4, dropout=dropout)
        self.cafe_gate = nn.Linear(128, 64)
        
        # classifiers
        self.fc_eeg = nn.Linear(64, num_classes)
        self.fc_fnirs = nn.Linear(64, num_classes)
        self.fc_fusion = nn.Linear(64, num_classes)
        
        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w3 = nn.Parameter(torch.tensor(1.0))
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, eeg, nirs):
        batch_size = eeg.size(0)
        
        # EEG branch process
        x = F.elu(self.eeg_bn1(self.eeg_conv1(eeg)))
        x = self.eeg_drop1(x)
        x = F.elu(self.eeg_bn2(self.eeg_conv2(x)))
        x = self.eeg_drop2(x)
        eeg_feat = self.eeg_pool(x).view(batch_size, -1) 
        eeg_feat = self.eeg_fc(eeg_feat)
        
        # fNIRS branch process
        y = F.elu(self.nirs_bn1(self.nirs_conv1(nirs)))
        y = self.nirs_drop1(y)
        y = F.elu(self.nirs_bn2(self.nirs_conv2(y)))
        y = self.nirs_drop2(y)
        
        y_spatial = self.nirs_spatial_pool(y).squeeze(-1).squeeze(-1) 
        nirs_feat_raw = self.nirs_time_pool(y_spatial).view(batch_size, -1) 
        nirs_feat = self.nirs_fc(nirs_feat_raw) 
        
        # EFGF Block
        eeg_p = self.efgf_eeg(eeg_feat)
        nirs_p = self.efgf_nirs(nirs_feat)
        alpha = torch.clamp(self.alpha, 0, 1)
        hybrid = alpha * eeg_p + (1 - alpha) * nirs_p
        gate = torch.sigmoid(self.efgf_gate(hybrid))
        fused_feat = eeg_feat + gate * nirs_feat 
        
        # CAFE Block
        nirs_seq = y_spatial.transpose(1, 2) 
        nirs_seq_proj = nn.Linear(32, 64).to(nirs.device)(nirs_seq) 
        nirs_proj = self.pos_encoder(nirs_seq_proj) 
        
        fused_seq = fused_feat.unsqueeze(1) 
        enhanced = self.cafe_attn(fused_seq, nirs_proj, nirs_proj).squeeze(1) 
        
        gate2 = torch.sigmoid(self.cafe_gate(torch.cat([fused_feat, enhanced], dim=-1)))
        final_feat = gate2 * enhanced + (1 - gate2) * fused_feat
        
        eeg_out = self.fc_eeg(self.dropout(eeg_feat))
        fnirs_out = self.fc_fnirs(self.dropout(nirs_feat))
        fusion_out = self.fc_fusion(self.dropout(final_feat))
        
        w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)
        final_out = w[0] * eeg_out + w[1] * fnirs_out + w[2] * fusion_out
        
        return final_out, eeg_out, fnirs_out, fusion_out

# MAIN PIPELINE 
if __name__ == "__main__":
    
    train_dataset = MMapHybridDataset(TRAIN_SUBS, "Train_Set")
    val_dataset = MMapHybridDataset(VAL_SUBS, "Validation_Set")
    test_dataset = MMapHybridDataset(TEST_SUBS, "Test_Set")

    # Dtaloaders
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=0)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = TSFNet_With_fNIRS_2D(eeg_ch=28, num_classes=3, dropout=0.5).to(device)

    print(f"Model Initialized. Total Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Running on Device: {device}\n")

    # optimizer and parameters
    optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=5e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    epochs = 100

    print("STARTING TRAINING")
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss, train_correct = 0.0, 0
        
        for eeg_b, nirs_b, lbl_b in train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            
            optimizer.zero_grad()
            pred, _, _, _ = model(eeg_b, nirs_b)
            
            loss = criterion(pred, lbl_b)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss.item()
            train_correct += (pred.argmax(1) == lbl_b).sum().item()
            
        train_acc = train_correct / len(train_loader.dataset)
        
        # validation
        model.eval()
        val_correct = 0
        with torch.no_grad():
            for eeg_b, nirs_b, lbl_b in val_loader:
                eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
                pred, _, _, _ = model(eeg_b, nirs_b)
                val_correct += (pred.argmax(1) == lbl_b).sum().item()
                
        val_acc = val_correct / len(val_loader.dataset)
        scheduler.step(val_acc)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth'))
            
        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{epochs} | Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc*100:.2f}% | Val Acc: {val_acc*100:.2f}% | Best Val: {best_val_acc*100:.2f}%")

    # FINAL MODEL TEST & EVALUATION
    print("\n RUNNING FINAL TEST EVALUATION")
    model.load_state_dict(torch.load(os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet.pth')))
    model.eval()

    all_preds, all_true = [], []
    with torch.no_grad():
        for eeg_b, nirs_b, lbl_b in test_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            pred, _, _, _ = model(eeg_b, nirs_b)
            all_preds.extend(pred.argmax(1).cpu().numpy())
            all_true.extend(lbl_b.cpu().numpy())

    print(f"\nOverall Model Test Accuracy: {accuracy_score(all_true, all_preds)*100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(all_true, all_preds, target_names=['0-back', '2-back', '3-back']))

    train_dataset.cleanup()
    val_dataset.cleanup()
    test_dataset.cleanup()
    print("\nCleanup completed. Temporary disk files removed.")

Scanning Train_Set - 18 subjects on Drive D...
VP016 Skipped - Insufficient windows (1026 < 2000). Check task markers!
VP003 Skipped - Insufficient windows (1026 < 2000). Check task markers!
VP011 Skipped - Insufficient windows (1023 < 2000). Check task markers!
VP010 Skipped - Insufficient windows (1023 < 2000). Check task markers!
VP021 Skipped - Insufficient windows (1017 < 2000). Check task markers!
VP012 Skipped - Insufficient windows (1026 < 2000). Check task markers!
VP022 Skipped - Insufficient windows (1026 < 2000). Check task markers!
VP025 Skipped - Insufficient windows (1023 < 2000). Check task markers!
VP017 Skipped - Insufficient windows (1026 < 2000). Check task markers!
VP014 Skipped - Insufficient windows (1016 < 2000). Check task markers!
VP005 Skipped - Insufficient windows (1016 < 2000). Check task markers!
VP013 Skipped - Insufficient windows (1023 < 2000). Check task markers!
VP024 Skipped - Insufficient windows (1026 < 2000). Check task markers!
VP020 Skipped - I

ValueError: No valid preprocessed data found for Train_Set after filtering!

In [ ]:


# TSFNet Revised (matched the paper, except for the data structure-related sections)- no opt

import os
import gc
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
import warnings
warnings.filterwarnings('ignore')

# CONFIGURATION
PROCESSED_DATA_PATH = r"D:\Research\EEG-fNIRS\project\project\Results\MULTIMODAL_PREPROCESSED"

ALL_SUBJECTS = list(range(1, 27))
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15

random.seed(42)
shuffled_subjects = ALL_SUBJECTS.copy()
random.shuffle(shuffled_subjects)

n_total = len(shuffled_subjects)
n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VAL_RATIO)
n_test = n_total - n_train - n_val

TRAIN_SUBS = shuffled_subjects[:n_train]
VAL_SUBS = shuffled_subjects[n_train:n_train + n_val]
TEST_SUBS = shuffled_subjects[n_train + n_val:]

print(f"Train: {len(TRAIN_SUBS)} subjects")
print(f"Val: {len(VAL_SUBS)} subjects")
print(f"Test: {len(TEST_SUBS)} subjects")

# DATASET
class MMapHybridDataset(Dataset):
    def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=2000):
        super().__init__()
        self.dataset_name = dataset_name
        self.valid_files = []
        self.total_windows = 0
        
        print(f"Scanning {dataset_name} - {len(subs_list)} subjects...")
        
        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
            
            if not os.path.exists(sub_folder):
                continue
                
            eeg_file = None
            nirs_file = None
            labels_file = None
            
            for f in os.listdir(sub_folder):
                if '_eeg_windows.npy' in f or '_eeg.npy' in f:
                    eeg_file = os.path.join(sub_folder, f)
                if '_fnirs_windows.npy' in f or '_fnirs.npy' in f:
                    nirs_file = os.path.join(sub_folder, f)
                if '_labels.npy' in f:
                    labels_file = os.path.join(sub_folder, f)
            
            if eeg_file and nirs_file and labels_file:
                try:
                    lbls = np.load(labels_file).astype(np.int64)
                    n_windows = lbls.shape[0]
                    
                    if n_windows < min_windows_threshold:
                        print(f"  {subject} Skipped - {n_windows} < {min_windows_threshold}")
                        continue
                    
                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject}: {n_windows} windows")
                    
                except Exception as e:
                    print(f"Error {subject}: {str(e)}")
                    continue
                    
        if self.total_windows == 0:
            raise ValueError(f"No valid data found for {dataset_name}!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)
        
        self.mmap_eeg_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_e.npy")
        self.mmap_nirs_path = os.path.join(PROCESSED_DATA_PATH, f"t_mmap_{dataset_name[:3]}_n.npy")
        
        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                try: os.remove(f_path)
                except: pass
            with open(f_path, 'wb') as f:
                pass 
        
        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', 
                               shape=(self.total_windows, 28, 600))
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', 
                               shape=(self.total_windows, 2, 100, 16, 16))
        self.y = np.zeros(self.total_windows, dtype='int64')
        
        print(f"streaming data to disk ({dataset_name})...")
        for f_info in self.valid_files:
            s, e = f_info['start_idx'], f_info['end_idx']
            self.X_eeg[s:e] = np.load(f_info['eeg_path'])
            self.X_nirs[s:e] = np.load(f_info['nirs_path'])
            self.y[s:e] = f_info['labels']
            gc.collect()
            
        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"  {dataset_name} Ready! Total: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor
        
    def cleanup(self):
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try: os.remove(f)
                except: pass

# POSITIONAL ENCODING
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): 
        return x + self.pe[:, :x.size(1), :]

# CROSS ATTENTION
class CrossAttention(nn.Module):
    def __init__(self, d_model, nhead=4, dropout=0.3):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, query, key, value):
        out, _ = self.attn(query, key, value)
        out = self.dropout(out)
        return self.norm(query + out)

# PEARSON CORRELATION for loss
def pearson_correlation(x, y):
    x = x.view(x.size(0), -1)
    y = y.view(y.size(0), -1)
    min_dim = min(x.size(1), y.size(1))
    x = x[:, :min_dim]
    y = y[:, :min_dim]
    
    x_mean = x.mean(dim=1, keepdim=True)
    y_mean = y.mean(dim=1, keepdim=True)
    x_centered = x - x_mean
    y_centered = y - y_mean
    
    numerator = (x_centered * y_centered).sum(dim=1)
    denominator = torch.sqrt((x_centered ** 2).sum(dim=1) * (y_centered ** 2).sum(dim=1) + 1e-8)
    
    return (numerator / denominator).mean()

# TSFNet 
class TSFNet_Complete(nn.Module):
    def __init__(self, eeg_ch=28, num_classes=3, dropout=0.3):
        super(TSFNet_Complete, self).__init__()
        
        # EEG Branch
        self.eeg_conv1 = nn.Conv1d(eeg_ch, 32, kernel_size=12, stride=6)
        self.eeg_bn1 = nn.BatchNorm1d(32)
        self.eeg_drop1 = nn.Dropout1d(dropout)
        
        self.eeg_conv2 = nn.Conv1d(32, 64, kernel_size=6, stride=2)
        self.eeg_bn2 = nn.BatchNorm1d(64)
        self.eeg_drop2 = nn.Dropout1d(dropout)
        
        self.eeg_pool = nn.AdaptiveAvgPool1d(30)
        self.eeg_fc = nn.Linear(64, 64)
        
        # fNIRS Branch
        self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        self.nirs_bn1 = nn.BatchNorm3d(16)
        self.nirs_drop1 = nn.Dropout3d(dropout)
        
        self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        self.nirs_bn2 = nn.BatchNorm3d(32)
        self.nirs_drop2 = nn.Dropout3d(dropout)
        
        self.nirs_spatial_pool = nn.AdaptiveAvgPool3d((None, 1, 1))
        self.nirs_time_pool = nn.AdaptiveAvgPool1d(1)
        self.nirs_fc = nn.Linear(32, 64)
        
        # Fusion Branch 
        self.fusion_conv1 = nn.Conv1d(eeg_ch, 32, kernel_size=12, stride=6)
        self.fusion_bn1 = nn.BatchNorm1d(32)
        self.fusion_drop1 = nn.Dropout1d(dropout)
        
        self.fusion_conv2 = nn.Conv1d(32, 64, kernel_size=6, stride=2)
        self.fusion_bn2 = nn.BatchNorm1d(64)
        self.fusion_drop2 = nn.Dropout1d(dropout)
        
        self.fusion_pool = nn.AdaptiveAvgPool1d(30)
        self.fusion_fc = nn.Linear(64, 64)
        
        # EFGF Layer
        self.efgf_conv_eeg = nn.Conv1d(64, 1, kernel_size=1)
        self.efgf_conv_fnirs = nn.Conv3d(32, 1, kernel_size=1)
        
        self.alpha = nn.Parameter(torch.tensor(0.5))
        self.gamma = nn.Parameter(torch.tensor(0.5))
        
        self.efgf_gap = nn.AdaptiveAvgPool1d(1)
        
        # CAFE Layer
        self.cafe_proj = nn.Linear(64, 64)
        self.pos_encoder = PositionalEncoding(64)
        self.cafe_attn = CrossAttention(64, nhead=4, dropout=dropout)
        self.cafe_gate = nn.Linear(128, 64)
        
        # Classifiers
        self.fc_eeg = nn.Linear(64, num_classes)
        self.fc_fnirs = nn.Linear(64, num_classes)
        self.fc_fusion = nn.Linear(64, num_classes)
        
        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w3 = nn.Parameter(torch.tensor(1.0))
        
        self.dropout = nn.Dropout(dropout)
        
        self.efgf_attn_map = None
        
    def forward(self, eeg, nirs, return_attn=False):
        batch_size = eeg.size(0)
        
        # EEG Branch
        x = F.elu(self.eeg_bn1(self.eeg_conv1(eeg)))
        x = self.eeg_drop1(x)
        x = F.elu(self.eeg_bn2(self.eeg_conv2(x)))
        x = self.eeg_drop2(x)
        eeg_seq = self.eeg_pool(x)
        eeg_feat = eeg_seq.mean(dim=2)  
        eeg_feat = self.eeg_fc(eeg_feat)
        
        # fNIRS Branch
        y = F.elu(self.nirs_bn1(self.nirs_conv1(nirs)))
        y = self.nirs_drop1(y)
        y = F.elu(self.nirs_bn2(self.nirs_conv2(y)))
        y = self.nirs_drop2(y)
        
        y_spatial = self.nirs_spatial_pool(y).squeeze(-1).squeeze(-1)
        nirs_feat_raw = self.nirs_time_pool(y_spatial).view(batch_size, -1) 
        nirs_feat = self.nirs_fc(nirs_feat_raw)
        nirs_feat_map = y 
        
        # Fusion Branch
        z = F.elu(self.fusion_bn1(self.fusion_conv1(eeg)))
        z = self.fusion_drop1(z)
        z = F.elu(self.fusion_bn2(self.fusion_conv2(z)))
        z = self.fusion_drop2(z)
        z = self.fusion_pool(z)  
        fusion_feat = z.mean(dim=2)  
        fusion_feat = self.fusion_fc(fusion_feat)
        fusion_seq = z 

        # EFGF Layer
        eeg_attn = self.efgf_conv_eeg(eeg_seq)
        eeg_attn = self.efgf_gap(eeg_attn)  
        eeg_attn = torch.sigmoid(eeg_attn).view(batch_size, -1)  
        
        fnirs_attn = self.efgf_conv_fnirs(nirs_feat_map)  
        fnirs_attn = F.adaptive_avg_pool3d(fnirs_attn, (1, 1, 1))  
        fnirs_attn = torch.sigmoid(fnirs_attn).view(batch_size, -1)  
        
        # Hybrid Attention Map 
        alpha_clamped = torch.clamp(self.alpha, 0, 1)
        hybrid_attn = alpha_clamped * eeg_attn + (1 - alpha_clamped) * fnirs_attn 
        hybrid_attn = hybrid_attn.unsqueeze(1)  
        
        self.efgf_attn_map = hybrid_attn.detach()
        
        phi_fusion = hybrid_attn * fusion_feat
        
        gamma_clamped = torch.clamp(self.gamma, 0, 1)
        enhanced_fusion = gamma_clamped * eeg_feat + (1 - gamma_clamped) * fusion_feat + phi_fusion
        
        # CAFE Layer
        fusion_seq_proj = enhanced_fusion.unsqueeze(1)  
        nirs_seq_proj = nirs_feat.unsqueeze(1) 
        nirs_seq_proj = self.cafe_proj(nirs_seq_proj)  
        nirs_seq_proj = self.pos_encoder(nirs_seq_proj)
        
        # Cross-Attention
        enhanced, _ = self.cafe_attn(fusion_seq_proj, nirs_seq_proj, nirs_seq_proj)
        
        # Gated Fusion
        gate = torch.sigmoid(self.cafe_gate(torch.cat([fusion_seq_proj, enhanced], dim=-1)))
        final_seq = gate * enhanced + (1 - gate) * fusion_seq_proj
        final_feat = final_seq.squeeze(1)  
        
        # Classifiers
        eeg_out = self.fc_eeg(self.dropout(eeg_feat))
        fnirs_out = self.fc_fnirs(self.dropout(nirs_feat))
        fusion_out = self.fc_fusion(self.dropout(final_feat))
        
        # Weighted Decision Fusion 
        w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)
        final_out = w[0] * eeg_out + w[1] * fnirs_out + w[2] * fusion_out
        
        if return_attn:
            return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat, nirs_feat, hybrid_attn
        return final_out, eeg_out, fnirs_out, fusion_out

# COMPLETE LOSS FUNCTION
def tsfnet_complete_loss(pred, eeg_out, fnirs_out, fusion_out,
                         eeg_feat, nirs_feat, hybrid_attn,
                         labels, lambda_eeg=0.2, lambda_fnirs=0.2):

    ce_loss = nn.CrossEntropyLoss()
    
    # Classification Loss 
    L_pred = ce_loss(pred, labels
    L_eeg = ce_loss(eeg_out, labels)
    L_fnirs = ce_loss(fnirs_out, labels)
    L_fusion = ce_loss(fusion_out, labels)
    
    L_class = L_pred + lambda_eeg * L_eeg + lambda_fnirs * L_fnirs + L_fusion
    
    # EFGF Loss 
    eeg_feat_expanded = eeg_feat.unsqueeze(1)  
    hybrid_attn_expanded = hybrid_attn 
    
    hybrid_attn_repeated = hybrid_attn_expanded.repeat(1, 1, eeg_feat.size(1))  
    
    pcc_efgf = pearson_correlation(eeg_feat_expanded, hybrid_attn_repeated)
    L_efgf = 1 - torch.abs(pcc_efgf)
    
    # CAFE Loss 
    pcc_cafe = pearson_correlation(fusion_out, fnirs_out)
    L_cafe = 1 - torch.abs(pcc_cafe)
    
    # Total Loss 
    L_total = L_class + L_efgf + L_cafe
    
    return L_total, L_class, L_efgf, L_cafe

# MAIN PIPELINE
if __name__ == "__main__":
        
    # Load datasets
    train_dataset = MMapHybridDataset(TRAIN_SUBS, "Train_Set")
    val_dataset = MMapHybridDataset(VAL_SUBS, "Validation_Set")
    test_dataset = MMapHybridDataset(TEST_SUBS, "Test_Set")

    # Dataloaders
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=0)

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = TSFNet_Complete(eeg_ch=28, num_classes=3, dropout=0.3).to(device)

    print(f"\nModel Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Device: {device}\n")

    # Optimizer
    optimizer = optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)

    best_val_acc = 0.0
    epochs = 100

    print("STARTING TRAINING")

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_loss_class = 0.0
        train_loss_efgf = 0.0
        train_loss_cafe = 0.0
        
        for eeg_b, nirs_b, lbl_b in train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            
            optimizer.zero_grad()
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat, nirs_feat, hybrid_attn = model(eeg_b, nirs_b, return_attn=True)
            
            loss_total, loss_class, loss_efgf, loss_cafe = tsfnet_complete_loss(
                pred, eeg_out, fnirs_out, fusion_out,
                eeg_feat, nirs_feat, hybrid_attn,
                lbl_b
            )
            
            loss_total.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss_total.item()
            train_loss_class += loss_class.item()
            train_loss_efgf += loss_efgf.item()
            train_loss_cafe += loss_cafe.item()
            train_correct += (pred.argmax(1) == lbl_b).sum().item()
            
        train_acc = train_correct / len(train_loader.dataset)
        
        # Validation
        model.eval()
        val_correct = 0
        with torch.no_grad():
            for eeg_b, nirs_b, lbl_b in val_loader:
                eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
                pred, _, _, _ = model(eeg_b, nirs_b)
                val_correct += (pred.argmax(1) == lbl_b).sum().item()
                
        val_acc = val_correct / len(val_loader.dataset)
        scheduler.step(val_acc)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet_complete.pth'))
            
        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{epochs} | "
                  f"Loss: {train_loss/len(train_loader):.4f} | "
                  f"L_class: {train_loss_class/len(train_loader):.4f} | "
                  f"L_efgf: {train_loss_efgf/len(train_loader):.4f} | "
                  f"L_cafe: {train_loss_cafe/len(train_loader):.4f} | "
                  f"Train: {train_acc*100:.2f}% | "
                  f"Val: {val_acc*100:.2f}% | "
                  f"Best: {best_val_acc*100:.2f}%")

    # FINAL TEST
    print("FINAL TEST EVALUATION")
    
    model.load_state_dict(torch.load(os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet_complete.pth')))
    model.eval()

    all_preds, all_true = [], []
    with torch.no_grad():
        for eeg_b, nirs_b, lbl_b in test_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            pred, _, _, _ = model(eeg_b, nirs_b)
            all_preds.extend(pred.argmax(1).cpu().numpy())
            all_true.extend(lbl_b.cpu().numpy())

    print(f"\nOverall Model Test Accuracy: {accuracy_score(all_true, all_preds)*100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(all_true, all_preds, target_names=['0-back', '2-back', '3-back']))

    # Cleanup
    train_dataset.cleanup()
    val_dataset.cleanup()
    test_dataset.cleanup()
    print("\nCleanup completed.")

In [ ]:
!pip install optuna

In [ ]:
# TSFNet WITH OPTUNA OPT.

import os
import gc
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
import warnings
warnings.filterwarnings('ignore')
import optuna

# CONFIGURATION
PROCESSED_DATA_PATH = r"D:\Research\EEG-fNIRS\project\project\Results\MULTIMODAL_PREPROCESSED"

ALL_SUBJECTS = list(range(1, 27))
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15

# random.seed(42)
shuffled_subjects = ALL_SUBJECTS.copy()
random.shuffle(shuffled_subjects)

n_total = len(shuffled_subjects)
n_train = int(n_total * TRAIN_RATIO)
n_val = int(n_total * VAL_RATIO)
n_test = n_total - n_train - n_val

TRAIN_SUBS = shuffled_subjects[:n_train]
VAL_SUBS = shuffled_subjects[n_train:n_train + n_val]
TEST_SUBS = shuffled_subjects[n_train + n_val:]

print(f"Train: {len(TRAIN_SUBS)} subjects")
print(f"Val: {len(VAL_SUBS)} subjects")
print(f"Test: {len(TEST_SUBS)} subjects")

# DATASET CLASS
class MMapHybridDataset(Dataset):
    def __init__(self, subs_list, dataset_name="Dataset", min_windows_threshold=2000):
        super().__init__()
        self.dataset_name = dataset_name
        self.valid_files = []
        self.total_windows = 0
        
        print(f"scanning {dataset_name} - {len(subs_list)} subjects...")
        
        for sub_id in subs_list:
            sub_str = f"{sub_id:03d}"
            subject = f"VP{sub_str}"
            sub_folder = os.path.join(PROCESSED_DATA_PATH, subject)
            
            if not os.path.exists(sub_folder):
                continue
                
            eeg_file = None
            nirs_file = None
            labels_file = None
            
            for f in os.listdir(sub_folder):
                if '_eeg_windows.npy' in f or '_eeg.npy' in f:
                    eeg_file = os.path.join(sub_folder, f)
                if '_fnirs_windows.npy' in f or '_fnirs.npy' in f:
                    nirs_file = os.path.join(sub_folder, f)
                if '_labels.npy' in f:
                    labels_file = os.path.join(sub_folder, f)
            
            if eeg_file and nirs_file and labels_file:
                try:
                    lbls = np.load(labels_file).astype(np.int64)
                    n_windows = lbls.shape[0]
                    
                    if n_windows < min_windows_threshold:
                        print(f"{subject} Skipped - {n_windows} < {min_windows_threshold}")
                        continue
                    
                    self.valid_files.append({
                        'subject': subject,
                        'eeg_path': eeg_file,
                        'nirs_path': nirs_file,
                        'labels': lbls,
                        'start_idx': self.total_windows,
                        'end_idx': self.total_windows + n_windows
                    })
                    self.total_windows += n_windows
                    print(f"{subject}: {n_windows} windows")
                    
                except Exception as e:
                    print(f"Error {subject}: {str(e)}")
                    continue
                    
        if self.total_windows == 0:
            raise ValueError(f"NO valid data found for {dataset_name}!")

        os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)
        
        temp_dir = os.path.join(PROCESSED_DATA_PATH, "temp_mmap")
        os.makedirs(temp_dir, exist_ok=True)
        
        safe_name = dataset_name.replace(" ", "_").replace("(", "").replace(")", "")
        self.mmap_eeg_path = os.path.join(temp_dir, f"mmap_{safe_name}_eeg.npy")
        self.mmap_nirs_path = os.path.join(temp_dir, f"mmap_{safe_name}_nirs.npy")
        
        for f_path in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f_path):
                try: os.remove(f_path)
                except: pass
        
        self.X_eeg = np.memmap(self.mmap_eeg_path, dtype='float32', mode='w+', 
                               shape=(self.total_windows, 28, 600))
        self.X_nirs = np.memmap(self.mmap_nirs_path, dtype='float32', mode='w+', 
                               shape=(self.total_windows, 2, 100, 16, 16))
        self.y = np.zeros(self.total_windows, dtype='int64')
        
        print(f"streaming data to disk ({dataset_name})...")
        for f_info in self.valid_files:
            s, e = f_info['start_idx'], f_info['end_idx']
            self.X_eeg[s:e] = np.load(f_info['eeg_path'])
            self.X_nirs[s:e] = np.load(f_info['nirs_path'])
            self.y[s:e] = f_info['labels']
            gc.collect()
            
        self.X_eeg.flush()
        self.X_nirs.flush()
        print(f"{dataset_name} Ready! Total: {self.total_windows} | Labels: {np.bincount(self.y)}\n")

    def __len__(self):
        return self.total_windows

    def __getitem__(self, idx):
        eeg_tensor = torch.from_numpy(self.X_eeg[idx]).float()
        nirs_tensor = torch.from_numpy(self.X_nirs[idx]).float()
        label_tensor = torch.tensor(self.y[idx], dtype=torch.long)
        return eeg_tensor, nirs_tensor, label_tensor
        
    def cleanup(self):
        del self.X_eeg
        del self.X_nirs
        gc.collect()
        temp_dir = os.path.dirname(self.mmap_eeg_path)
        for f in [self.mmap_eeg_path, self.mmap_nirs_path]:
            if os.path.exists(f):
                try: os.remove(f)
                except: pass
        if os.path.exists(temp_dir):
            try: os.rmdir(temp_dir)
            except: pass

# POSITIONAL ENCODING
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): 
        return x + self.pe[:, :x.size(1), :]

# CROSS ATTENTION
class CrossAttention(nn.Module):
    def __init__(self, d_model, nhead=4, dropout=0.3):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, query, key, value):
        out, _ = self.attn(query, key, value)
        out = self.dropout(out)
        return self.norm(query + out)

# PEARSON CORRELATION
def pearson_correlation(x, y):
    x = x.view(x.size(0), -1)
    y = y.view(y.size(0), -1)
    min_dim = min(x.size(1), y.size(1))
    x = x[:, :min_dim]
    y = y[:, :min_dim]
    
    x_mean = x.mean(dim=1, keepdim=True)
    y_mean = y.mean(dim=1, keepdim=True)
    x_centered = x - x_mean
    y_centered = y - y_mean
    
    numerator = (x_centered * y_centered).sum(dim=1)
    denominator = torch.sqrt((x_centered ** 2).sum(dim=1) * (y_centered ** 2).sum(dim=1) + 1e-8)
    
    return (numerator / denominator).mean()

# TSFNet MODEL
class TSFNet_Complete(nn.Module):
    def __init__(self, eeg_ch=28, num_classes=3, dropout=0.3):
        super(TSFNet_Complete, self).__init__()
        
        # EEG Branch
        self.eeg_conv1 = nn.Conv1d(eeg_ch, 32, kernel_size=12, stride=6)
        self.eeg_bn1 = nn.BatchNorm1d(32)
        self.eeg_drop1 = nn.Dropout1d(dropout)
        
        self.eeg_conv2 = nn.Conv1d(32, 64, kernel_size=6, stride=2)
        self.eeg_bn2 = nn.BatchNorm1d(64)
        self.eeg_drop2 = nn.Dropout1d(dropout)
        
        self.eeg_pool = nn.AdaptiveAvgPool1d(30)
        self.eeg_fc = nn.Linear(64, 64)
        
        # fNIRS Branch
        self.nirs_conv1 = nn.Conv3d(2, 16, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        self.nirs_bn1 = nn.BatchNorm3d(16)
        self.nirs_drop1 = nn.Dropout3d(dropout)
        
        self.nirs_conv2 = nn.Conv3d(16, 32, kernel_size=(3, 3, 3), stride=(1, 2, 2), padding=(1, 1, 1))
        self.nirs_bn2 = nn.BatchNorm3d(32)
        self.nirs_drop2 = nn.Dropout3d(dropout)
        
        self.nirs_spatial_pool = nn.AdaptiveAvgPool3d((None, 1, 1))
        self.nirs_time_pool = nn.AdaptiveAvgPool1d(1)
        self.nirs_fc = nn.Linear(32, 64)
        
        # Fusion Branch
        self.fusion_conv1 = nn.Conv1d(eeg_ch, 32, kernel_size=12, stride=6)
        self.fusion_bn1 = nn.BatchNorm1d(32)
        self.fusion_drop1 = nn.Dropout1d(dropout)
        
        self.fusion_conv2 = nn.Conv1d(32, 64, kernel_size=6, stride=2)
        self.fusion_bn2 = nn.BatchNorm1d(64)
        self.fusion_drop2 = nn.Dropout1d(dropout)
        
        self.fusion_pool = nn.AdaptiveAvgPool1d(30)
        self.fusion_fc = nn.Linear(64, 64)
        
        # EFGF Layer
        self.efgf_conv_eeg = nn.Conv1d(64, 1, kernel_size=1)
        self.efgf_conv_fnirs = nn.Conv3d(32, 1, kernel_size=1)
        
        self.alpha = nn.Parameter(torch.tensor(0.5))
        self.gamma = nn.Parameter(torch.tensor(0.5))
        
        self.efgf_gap = nn.AdaptiveAvgPool1d(1)
        
        # CAFE Layer
        self.cafe_proj = nn.Linear(64, 64)
        self.pos_encoder = PositionalEncoding(64)
        self.cafe_attn = CrossAttention(64, nhead=4, dropout=dropout)
        self.cafe_gate = nn.Linear(128, 64)
        
        # Classifiers
        self.fc_eeg = nn.Linear(64, num_classes)
        self.fc_fnirs = nn.Linear(64, num_classes)
        self.fc_fusion = nn.Linear(64, num_classes)
        
        self.w1 = nn.Parameter(torch.tensor(1.0))
        self.w2 = nn.Parameter(torch.tensor(1.0))
        self.w3 = nn.Parameter(torch.tensor(1.0))
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, eeg, nirs, return_attn=False):
        batch_size = eeg.size(0)
        
        # EEG
        x = F.elu(self.eeg_bn1(self.eeg_conv1(eeg)))
        x = self.eeg_drop1(x)
        x = F.elu(self.eeg_bn2(self.eeg_conv2(x)))
        x = self.eeg_drop2(x)
        eeg_seq = self.eeg_pool(x)
        eeg_feat = eeg_seq.mean(dim=2)
        eeg_feat = self.eeg_fc(eeg_feat)
        
        # fNIRS
        y = F.elu(self.nirs_bn1(self.nirs_conv1(nirs)))
        y = self.nirs_drop1(y)
        y = F.elu(self.nirs_bn2(self.nirs_conv2(y)))
        y = self.nirs_drop2(y)
        
        y_spatial = self.nirs_spatial_pool(y).squeeze(-1).squeeze(-1)
        nirs_feat_raw = self.nirs_time_pool(y_spatial).view(batch_size, -1)
        nirs_feat = self.nirs_fc(nirs_feat_raw)
        nirs_feat_map = y
        
        # Fusion
        z = F.elu(self.fusion_bn1(self.fusion_conv1(eeg)))
        z = self.fusion_drop1(z)
        z = F.elu(self.fusion_bn2(self.fusion_conv2(z)))
        z = self.fusion_drop2(z)
        z = self.fusion_pool(z)
        fusion_feat = z.mean(dim=2)
        fusion_feat = self.fusion_fc(fusion_feat)
        fusion_seq = z

        # EFGF
        eeg_attn = self.efgf_conv_eeg(eeg_seq)
        eeg_attn = self.efgf_gap(eeg_attn)
        eeg_attn = torch.sigmoid(eeg_attn).view(batch_size, -1)
        
        fnirs_attn = self.efgf_conv_fnirs(nirs_feat_map)
        fnirs_attn = F.adaptive_avg_pool3d(fnirs_attn, (1, 1, 1))
        fnirs_attn = torch.sigmoid(fnirs_attn).view(batch_size, -1)
        
        alpha_clamped = torch.clamp(self.alpha, 0, 1)
        hybrid_attn = alpha_clamped * eeg_attn + (1 - alpha_clamped) * fnirs_attn
        hybrid_attn = hybrid_attn.unsqueeze(1)
        
        phi_fusion = hybrid_attn * fusion_feat
        
        gamma_clamped = torch.clamp(self.gamma, 0, 1)
        enhanced_fusion = gamma_clamped * eeg_feat + (1 - gamma_clamped) * fusion_feat + phi_fusion
        
        # CAFE
        fusion_seq_proj = enhanced_fusion.unsqueeze(1)
        nirs_seq_proj = nirs_feat.unsqueeze(1)
        nirs_seq_proj = self.cafe_proj(nirs_seq_proj)
        nirs_seq_proj = self.pos_encoder(nirs_seq_proj)
        
        enhanced, _ = self.cafe_attn(fusion_seq_proj, nirs_seq_proj, nirs_seq_proj)
        
        gate = torch.sigmoid(self.cafe_gate(torch.cat([fusion_seq_proj, enhanced], dim=-1)))
        final_seq = gate * enhanced + (1 - gate) * fusion_seq_proj
        final_feat = final_seq.squeeze(1)
        
        # Classifiers
        eeg_out = self.fc_eeg(self.dropout(eeg_feat))
        fnirs_out = self.fc_fnirs(self.dropout(nirs_feat))
        fusion_out = self.fc_fusion(self.dropout(final_feat))
        
        w = F.softmax(torch.stack([self.w1, self.w2, self.w3]), dim=0)
        final_out = w[0] * eeg_out + w[1] * fnirs_out + w[2] * fusion_out
        
        if return_attn:
            return final_out, eeg_out, fnirs_out, fusion_out, eeg_feat, nirs_feat, hybrid_attn
        return final_out, eeg_out, fnirs_out, fusion_out

def tsfnet_complete_loss(pred, eeg_out, fnirs_out, fusion_out,
                         eeg_feat, nirs_feat, hybrid_attn,
                         labels, lambda_eeg=0.2, lambda_fnirs=0.2):
    
    ce_loss = nn.CrossEntropyLoss()
    
    L_pred = ce_loss(pred, labels)
    L_eeg = ce_loss(eeg_out, labels)
    L_fnirs = ce_loss(fnirs_out, labels)
    L_fusion = ce_loss(fusion_out, labels)
    
    L_class = L_pred + lambda_eeg * L_eeg + lambda_fnirs * L_fnirs + L_fusion
    
    eeg_feat_expanded = eeg_feat.unsqueeze(1)
    hybrid_attn_expanded = hybrid_attn
    hybrid_attn_repeated = hybrid_attn_expanded.repeat(1, 1, eeg_feat.size(1))
    
    pcc_efgf = pearson_correlation(eeg_feat_expanded, hybrid_attn_repeated)
    L_efgf = 1 - torch.abs(pcc_efgf)
    
    pcc_cafe = pearson_correlation(fusion_out, fnirs_out)
    L_cafe = 1 - torch.abs(pcc_cafe)
    
    L_total = L_class + L_efgf + L_cafe
    
    return L_total, L_class, L_efgf, L_cafe

# OPTUNA HYPERPARAMETER OPT.
def objective(trial, train_dataset, val_dataset):
    
    batch_size = trial.suggest_categorical('batch_size', [32, 64])
    lr = trial.suggest_float('lr', 1e-4, 1e-3, log=True)
    dropout = trial.suggest_float('dropout', 0.2, 0.6)
    weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)
    lambda_eeg = trial.suggest_float('lambda_eeg', 0.05, 0.5)
    lambda_fnirs = trial.suggest_float('lambda_fnirs', 0.05, 0.5)
    nhead = trial.suggest_categorical('nhead', [2, 4, 8])
    d_model = trial.suggest_categorical('d_model', [32, 64, 128])
    
    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = TSFNet_Complete(eeg_ch=28, num_classes=3, dropout=dropout).to(device)
    
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
    
    best_val_acc = 0.0
    epochs = 30  
    
    for epoch in range(1, epochs + 1):
        model.train()
        train_correct = 0
        
        for eeg_b, nirs_b, lbl_b in train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            
            optimizer.zero_grad()
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat, nirs_feat, hybrid_attn = model(
                eeg_b, nirs_b, return_attn=True
            )
            
            loss_total, _, _, _ = tsfnet_complete_loss(
                pred, eeg_out, fnirs_out, fusion_out,
                eeg_feat, nirs_feat, hybrid_attn, lbl_b,
                lambda_eeg, lambda_fnirs
            )
            
            loss_total.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_correct += (pred.argmax(1) == lbl_b).sum().item()
            
        train_acc = train_correct / len(train_loader.dataset)
        
        model.eval()
        val_correct = 0
        with torch.no_grad():
            for eeg_b, nirs_b, lbl_b in val_loader:
                eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
                pred, _, _, _ = model(eeg_b, nirs_b)
                val_correct += (pred.argmax(1) == lbl_b).sum().item()
                
        val_acc = val_correct / len(val_loader.dataset)
        scheduler.step(val_acc)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
    
    return best_val_acc

def run_optuna_tuning(train_dataset, val_dataset, n_trials=30):
    print("OPTUNA HYPERPARAMETER OPTIMIZATION")
    print(f"Number of trials: {n_trials}")
    
    # Create study
    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(),
        pruner=optuna.pruners.MedianPruner()
    )
    
    # Run optimization
    study.optimize(
        lambda trial: objective(trial, train_dataset, val_dataset),
        n_trials=n_trials,
        show_progress_bar=True
    )
    
    # Best results
    print("BEST HYPERPARAMETERS FOUND")
    print(f"Best Validation Accuracy: {study.best_value*100:.2f}%")
    print("\nBest Parameters:")
    for key, value in study.best_params.items():
        print(f"  {key}: {value}")
    
    return study.best_params, study.best_value

# MAIN PIPELINE
if __name__ == "__main__":
    
    # LOAD DATA
    print("\nLOADING DATA")
    
    train_dataset = MMapHybridDataset(TRAIN_SUBS, "Train_Set", min_windows_threshold=2000)
    val_dataset = MMapHybridDataset(VAL_SUBS, "Validation_Set", min_windows_threshold=2000)
    test_dataset = MMapHybridDataset(TEST_SUBS, "Test_Set", min_windows_threshold=2000)
    
    # oPTUNA HYPERPARAMETER TUNING
    best_params, best_score = run_optuna_tuning(train_dataset, val_dataset, n_trials=30)
    
    # USE BEST PARAMETERS
   # best_params = {
   #     'batch_size': 64,
   #     'lr': 0.0008,
   #     'dropout': 0.35,
   #     'weight_decay': 0.0002,
   #     'lambda_eeg': 0.25,
   #     'lambda_fnirs': 0.20,
   #     'nhead': 4,
   #     'd_model': 64
   # }
    
    for key, value in best_params.items():
        print(f"  {key}: {value}")
    
    # TRAIN 
    BATCH_SIZE = best_params['batch_size']
    LR = best_params['lr']
    DROPOUT = best_params['dropout']
    WEIGHT_DECAY = best_params['weight_decay']
    LAMBDA_EEG = best_params['lambda_eeg']
    LAMBDA_FNIRS = best_params['lambda_fnirs']
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = TSFNet_Complete(eeg_ch=28, num_classes=3, dropout=DROPOUT).to(device)
    
    print(f"\nModel Parameters: {sum(p.numel() for p in model.parameters()):,}")
    print(f"Device: {device}\n")
    
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=10)
    
    best_val_acc = 0.0
    epochs = 100
    
    print("STARTING TRAINING")
    
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_loss_class = 0.0
        train_loss_efgf = 0.0
        train_loss_cafe = 0.0
        
        for eeg_b, nirs_b, lbl_b in train_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            
            optimizer.zero_grad()
            pred, eeg_out, fnirs_out, fusion_out, eeg_feat, nirs_feat, hybrid_attn = model(
                eeg_b, nirs_b, return_attn=True
            )
            
            loss_total, loss_class, loss_efgf, loss_cafe = tsfnet_complete_loss(
                pred, eeg_out, fnirs_out, fusion_out,
                eeg_feat, nirs_feat, hybrid_attn, lbl_b,
                LAMBDA_EEG, LAMBDA_FNIRS
            )
            
            loss_total.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            train_loss += loss_total.item()
            train_loss_class += loss_class.item()
            train_loss_efgf += loss_efgf.item()
            train_loss_cafe += loss_cafe.item()
            train_correct += (pred.argmax(1) == lbl_b).sum().item()
            
        train_acc = train_correct / len(train_loader.dataset)
        
        model.eval()
        val_correct = 0
        with torch.no_grad():
            for eeg_b, nirs_b, lbl_b in val_loader:
                eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
                pred, _, _, _ = model(eeg_b, nirs_b)
                val_correct += (pred.argmax(1) == lbl_b).sum().item()
                
        val_acc = val_correct / len(val_loader.dataset)
        scheduler.step(val_acc)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet_optuna.pth'))
            
        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:3d}/{epochs} | "
                  f"Loss: {train_loss/len(train_loader):.4f} | "
                  f"L_class: {train_loss_class/len(train_loader):.4f} | "
                  f"L_efgf: {train_loss_efgf/len(train_loader):.4f} | "
                  f"L_cafe: {train_loss_cafe/len(train_loader):.4f} | "
                  f"Train: {train_acc*100:.2f}% | "
                  f"Val: {val_acc*100:.2f}% | "
                  f"Best: {best_val_acc*100:.2f}%")
    
    # FINAL TEST
    print("FINAL TEST EVALUATION")
    
    model.load_state_dict(torch.load(os.path.join(PROCESSED_DATA_PATH, 'best_tsfnet_optuna.pth')))
    model.eval()
    
    all_preds, all_true = [], []
    with torch.no_grad():
        for eeg_b, nirs_b, lbl_b in test_loader:
            eeg_b, nirs_b, lbl_b = eeg_b.to(device), nirs_b.to(device), lbl_b.to(device)
            pred, _, _, _ = model(eeg_b, nirs_b)
            all_preds.extend(pred.argmax(1).cpu().numpy())
            all_true.extend(lbl_b.cpu().numpy())
    
    print(f"\nOverall Model TEst Accuracy: {accuracy_score(all_true, all_preds)*100:.2f}%")
    print("\nClassification Report:")
    print(classification_report(all_true, all_preds, target_names=['0-back', '2-back', '3-back']))
    
    # CLEANUP
    train_dataset.cleanup()
    val_dataset.cleanup()
    test_dataset.cleanup()
    
    print("ALL COMPLETED SUCCESSFULLY!")